In [3]:


def changecase(func):
    def myinner(*args, **kwargs):
        return func(*args, **kwargs).upper()
        
    return myinner
    
    
@changecase   
def myfunction(name):
    return f"Hello {name}"

print(changecase.__name__)
print(changecase.__doc__)
print(changecase.__module__)
print(changecase.__annotations__)
#print(changecase.__wrapped__)
changecase.fname="testfile"
print(changecase.__dict__)

myfunction("chen")

changecase
None
__main__
{}
{'fname': 'testfile'}


'HELLO CHEN'

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# measure_training_time(func):
functools.wraps ist sehr wichtig, damit:

Funktionsname

Docstring

Signatur
erhalten bleiben (z.B. für ML-Frameworks & Debugging)


In [5]:
import time
import functools

def measure_training_time(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()

        #print(f"[HOOK] {func.__name__} took {end - start:.4f} seconds")
        return result

    return wrapper

In [6]:
#Dummy-Daten
#X = torch.randn(1000, 10)
#y = torch.randn(1000, 1)

X = torch.randn(1000, 10)
true_w = torch.randn(10, 1)

y = X @ true_w + 0.1 * torch.randn(1000, 1)

# was ist dataset?
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32)

print("X.shape:", X.shape)
print("dataset:\n", type(dataset),"\n", dataset[0:2])
print("dataloader:\n", type(dataloader),"\n", dataloader)

X.shape: torch.Size([1000, 10])
dataset:
 <class 'torch.utils.data.dataset.TensorDataset'> 
 (tensor([[-0.0741,  0.1415,  0.0792,  1.2009,  0.2122, -0.9504, -0.8003, -3.0951,
          0.6353, -0.1528],
        [ 0.4150, -0.2489, -0.5497, -0.2715, -1.1132, -0.3691, -0.8488,  2.1458,
          1.7772, -1.4737]]), tensor([[-2.1776],
        [-0.2199]]))
dataloader:
 <class 'torch.utils.data.dataloader.DataLoader'> 


In [46]:
"""model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)
"""

model = nn.Sequential(
    nn.Linear(10, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()


In [47]:
@measure_training_time
def train_one_epoch(model, dataloader, optimizer, loss_fn):
    model.train()
    total_loss = 0.0

    for x, y in dataloader:
        optimizer.zero_grad()
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [48]:
# loss = train_one_epoch(model, dataloader, optimizer, loss_fn)
# print("Loss:", loss)

num_epochs = 500
loss_history = []

for epoch in range(1, num_epochs + 1):
    loss = train_one_epoch(model, dataloader, optimizer, loss_fn)
    loss_history.append(loss)

    if epoch % 20 == 0:
        print(f"Epoch [{epoch}/{num_epochs}] - Loss: {loss:.4f}")


Epoch [20/500] - Loss: 0.0215
Epoch [40/500] - Loss: 0.0120
Epoch [60/500] - Loss: 0.0079
Epoch [80/500] - Loss: 0.0059
Epoch [100/500] - Loss: 0.0047
Epoch [120/500] - Loss: 0.0039
Epoch [140/500] - Loss: 0.0032
Epoch [160/500] - Loss: 0.0043
Epoch [180/500] - Loss: 0.0045
Epoch [200/500] - Loss: 0.0035
Epoch [220/500] - Loss: 0.0031
Epoch [240/500] - Loss: 0.0041
Epoch [260/500] - Loss: 0.0272
Epoch [280/500] - Loss: 0.0040
Epoch [300/500] - Loss: 0.0025
Epoch [320/500] - Loss: 0.0018
Epoch [340/500] - Loss: 0.0015
Epoch [360/500] - Loss: 0.0034
Epoch [380/500] - Loss: 0.0021
Epoch [400/500] - Loss: 0.0019
Epoch [420/500] - Loss: 0.0090
Epoch [440/500] - Loss: 0.0032
Epoch [460/500] - Loss: 0.0018
Epoch [480/500] - Loss: 0.0012
Epoch [500/500] - Loss: 0.0009


In [54]:
# Vorhersage → ordinale Klasse

model.eval() # 1. Layer-Verhalten auf "Vorhersage" stellen
with torch.no_grad():
    y_pred = model(X)


mse = torch.mean((y_pred - y) ** 2)
rmse = torch.sqrt(mse)

print("MSE :", mse.item())
print("RMSE:", rmse.item())

ss_res = torch.sum((y - y_pred) ** 2)
ss_tot = torch.sum((y - y.mean()) ** 2)

r2 = 1 - ss_res / ss_tot
print("R²:", r2.item())

print("True:", y[:10])
print("Pred:", y_pred[:10])

MSE : 0.002518703928217292
RMSE: 0.05018668994307518
R²: 0.9996512532234192
True: tensor([[-0.8049],
        [ 1.0862],
        [ 1.5539],
        [ 0.6223],
        [-5.8251],
        [ 2.7543],
        [-5.6339],
        [ 2.2723],
        [ 1.3298],
        [-0.1479]])
Pred: tensor([[-0.8222],
        [ 1.0474],
        [ 1.5499],
        [ 0.6391],
        [-5.8580],
        [ 2.7731],
        [-5.6138],
        [ 2.2783],
        [ 1.4522],
        [-0.1474]])


In [55]:
loss_fn = torch.nn.MSELoss()

eval_loss = loss_fn(y_pred, y)
print("Eval MSE:", eval_loss.item())

Eval MSE: 0.002518703928217292
